In [2]:
import torch.nn as nn
import torch

In [3]:
def resnet18():
    return ResNet(BasicBlock, [2,2,2,2])

def resnet34():
    return ResNet(BasicBlock, [3,4,6,3])

def resnet50():
    return ResNet(BottleNeck, [3,4,6,3])

def resnet101():
    return ResNet(BottleNeck, [3,4,23,3])

def resnet152():
    return ResNet(BottleNeck, [3,8,36,3])

In [5]:
class ResNet(nn.Module):
    def __init__(self, block, num_block, num_classes=10, init_weights=True): # init_weights=True(가중치 초기화)
        super.__init__()
        self.in_channels= 64 # 모두 처음 채널은 64로 시작한다. 
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False), # 초반에 이미지 크기 확 줄이기(1/4)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        # block = Block의 개수
        self.conv2_x = self._make_layer(block, 64, num_block[0], 1) # 아마자 크기 유지
        self.conv3_x = self._make_layer(block, 128, num_block[1], 2) # 이미지 절반
        self.conv4_x = self._make_layer(block, 256, num_block[2], 2) # 이미지 절반
        self.conv5_x = self._make_layer(block, 512, num_block[3], 2) # 이미지 절반

        self.avg_pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512*block.expansion, num_classes)

        if init_weights:
            self._initialize_weights()

    # 하나의 ResNet layer를 생성하는 함수 (여러 개의 residual block을 쌓음)
    # 파라미터 : residual block 종류, layer 채널 수, block 개수, stride
    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides=[stride] + [1]*(num_blocks-1) # stride의 길이 = num_blocks의 길이 
        layers=[] # 생성된 block들을 순서대로 당을 리스트 초기화
        for stride in strides:
            # residual block 생성 후 리스트에 추가
            layers.append(block(self.in_channels, out_channels, stride))
            # 다음 block을 위해 입력 채널 수 업데이트 
            self.in_channels = out_channels*block.expansion
        # 리스트에 담긴 block들을 순차적으로 연결하여 하나의 layer로 반환 
        return nn.Sequential(*layers)


    def forward(self,x):
        output = self.conv1(x)
        output = self.conv2_x(output)
        x=self.conv3_x(output)
        x=self.conv4_x(x)
        x=self.conv5_x(x)
        x=self.avg_pool(x)
        x=x.view(x.size(0),-1)
        x=self.fc(x)
        return x
    
    # ResNet 모델의 가중치 초기화
    def _initialize_weights(self):
        for m in self.modules(): # 모델 안에 포함된 모든 layer/block 순회
            if isinstance(m, nn.Conv2d): # m이 Conv2d인지 확인 
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: # Conv layer에 bias가 있으면
                    nn.init.constant_(m.bias, 0) # bias=False로 초기화

            elif isinstance(m.BatchNorm2d): # 현재 순회 중인 모듈 m이 BatchNorm2d인지 확인
                nn.init.constant_(m.weight,1) # BatchNorm의 scale=1
                nn.init.constant_(m.bias,0) # BatchNorm의 bias=0
                
            elif isinstance(m, nn.Linear): # 현재 순회 중인 모듈 m이 Linear layer인지 확인
                nn.init.normal_(m.weight, 0, 0.01) # Linear weight를 정규 분포(normal)로 초기화 시키기
                nn.init.constant_(m.biax, 0) # bias=0으로 초기화

In [6]:
class BasicBlock(nn.Module):
    expansion=1
    def __init__(self, in_channels, out_channels, stride=1):
        super.__init__()

        self.residual_function = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels*BasicBlock.expansion, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_channels*BasicBlock.expansion)
        )
        self.shortcut = nn.Sequential()
        self.relu = nn.ReLU()

        # 이미지 크기가 다르거나, 채널 개수가 다르면
        if stride !=1 or in_channels != BasicBlock.expansion * out_channels:
            self.shortcut = nn.Sequential(
                # basic block에서는 downsample에서 kernel_size=1x1
                nn.Conv2d(in_channels, out_channels*BasicBlock.expansion, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels*BasicBlock.expansion)
            )

    def forward(self,x):
        x=self.residual_function(x) + self.shortcut(x)
        x=self.relu(x)
        return x

In [ ]:
class BottleNeck(nn.Module):
    expansion=4
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.residual_function = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels*BottleNeck.expansion, kernel_size=1, stride=1, bias=False),
            nn.BatchNorm2d(out_channels*BottleNeck.expansion)
        )
        self.shortcut = nn.Sequential()
        self.relu = nn.ReLU()

        if stride !=1 or in_channels != out_channels*BottleNeck.expansion:
            self.shortcut = nn.Sequential(
                # downsample의 경우 대부분 kernel=1x1
                nn.Conv2d(in_channels, out_channels*BottleNeck.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels*BottleNeck.expansion)
            )

    def forward(self,x):
        x=self.residual_function(x) + self.shortcut(x)
        x=self.relu(x)
        return x